#Feature Engineering Gold Prices Data (2016-2026)
resource: https://finance.yahoo.com/quote/GC%3DF/history/

##Import Library dan Upload Data

In [29]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import seaborn as sns
import matplotlib.pyplot as plt

In [30]:
price_df_fe = pd.read_csv("gold_price_yf.csv")
price_df_fe = price_df_fe.drop(columns=['Unnamed: 0'], errors='ignore')
price_df_fe.head()

,Date,Open,High,Low,Close,Volume
0,2016-05-19,1248.000000,1255.500000,1247.500000,1254.199951,69.0
1,2016-05-20,1256.599976,1256.599976,1252.400024,1252.400024,44.0
2,2016-05-23,1251.599976,1251.599976,1247.500000,1251.099976,56.0
3,2016-05-24,1240.000000,1240.000000,1228.199951,1228.900024,20.0
4,2016-05-25,1219.300049,1223.500000,1218.599976,1223.500000,5.0


In [31]:
price_df_fe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2513 entries, 0 to 2512
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    2513 non-null   object 
 1   Open    2513 non-null   float64
 2   High    2513 non-null   float64
 3   Low     2513 non-null   float64
 4   Close   2513 non-null   float64
 5   Volume  2513 non-null   float64
dtypes: float64(5), object(1)
memory usage: 117.9+ KB


##Tahap 1. Feature Engineering

In [32]:
# Ubah kolom 'Date' menjadi Datetime
price_df_fe['Date'] = pd.to_datetime(
    price_df_fe['Date']
)

price_df_fe = price_df_fe.sort_values(
    'Date'
)

In [33]:
# Lag Features
price_df_fe['Lag_1'] = price_df_fe['Close'].shift(1)
price_df_fe['Lag_7'] = price_df_fe['Close'].shift(7)
price_df_fe['Lag_30'] = price_df_fe['Close'].shift(30)

In [34]:
# Daily Return
price_df_fe['Return_1D'] = price_df_fe['Close'].pct_change()

In [35]:
# Moving Average (MA)
price_df_fe['MA_7'] = price_df_fe['Close'].rolling(7).mean()
price_df_fe['MA_30'] = price_df_fe['Close'].rolling(30).mean()

In [36]:
# Volatilitas
price_df_fe['Volatility_7'] = (
    price_df_fe['Close']
    .rolling(7)
    .std()
)

price_df_fe['Volatility_30'] = (
    price_df_fe['Close']
    .rolling(30)
    .std()
)

In [37]:
# High-Low Spread
price_df_fe['HL_Spread'] = (
    price_df_fe['High'] -
    price_df_fe['Low']
)

In [38]:
# Open-Close Different
price_df_fe['OC_Diff'] = (
    price_df_fe['Close'] -
    price_df_fe['Open']
)

In [39]:
# Moving Average (MA) dari Volume
price_df_fe['Volume_MA_7'] = (
    price_df_fe['Volume']
    .rolling(7)
    .mean()
)

price_df_fe['Volume_MA_30'] = (
    price_df_fe['Volume']
    .rolling(30)
    .mean()
)

In [40]:
# Exponential Moving Average
price_df_fe['EMA_7'] = (
    price_df_fe['Close']
    .ewm(span=7, adjust=False)
    .mean()
)
price_df_fe['EMA_30'] = (
    price_df_fe['Close']
    .ewm(span=7, adjust=False)
    .mean()
)

In [41]:
# Momentum
price_df_fe['Momentum_7'] = (
    price_df_fe['Close'] - price_df_fe['Close'].shift(7)
)
price_df_fe['Momentum_30'] = (
    price_df_fe['Close'] - price_df_fe['Close'].shift(30)
)

**Insight:**

- Mengubah kolom Date ke format datetime dan mengurutkan data berdasarkan tanggal untuk menjaga urutan temporal data time series.
- Membuat Lag Features (Lag_1, Lag_7, Lag_30) untuk menangkap pengaruh harga pada periode sebelumnya terhadap harga saat ini.
- Membuat Daily Return (Return_1D) untuk mengukur persentase perubahan harga harian.
- Membuat Moving Average (MA_7, MA_30) untuk mengidentifikasi tren harga jangka pendek dan menengah.
- Membuat Volatility (Volatility_7, Volatility_30) untuk mengukur tingkat fluktuasi harga dalam periode tertentu.
- Membuat High-Low Spread (HL_Spread) untuk menggambarkan rentang pergerakan harga harian.
- Membuat Open-Close Difference (OC_Diff) untuk menunjukkan selisih antara harga pembukaan dan penutupan dalam satu hari perdagangan.
- Membuat Volume Moving Average (Volume_MA_7, Volume_MA_30) untuk melihat tren aktivitas perdagangan berdasarkan volume transaksi.
- Membuat Exponential Moving Average (EMA_7, EMA_30) yang lebih responsif terhadap perubahan harga terbaru dibandingkan moving average biasa.
- Membuat Momentum (Momentum_7, Momentum_30) untuk mengukur kekuatan dan arah pergerakan harga dalam periode tertentu.
- Feature engineering menghasilkan fitur-fitur yang merepresentasikan aspek historis, tren, volatilitas, momentum, dan aktivitas pasar sehingga dapat memberikan informasi yang lebih kaya untuk proses pemodelan.

## Tahap 2. Drop Incomplete Features

In [42]:
# Hapus missing value
price_df_fe = price_df_fe.dropna()

In [43]:
# Cek missing value
print(price_df_fe.isnull().sum())

Date             0
Open             0
High             0
Low              0
Close            0
Volume           0
Lag_1            0
Lag_7            0
Lag_30           0
Return_1D        0
MA_7             0
MA_30            0
Volatility_7     0
Volatility_30    0
HL_Spread        0
OC_Diff          0
Volume_MA_7      0
Volume_MA_30     0
EMA_7            0
EMA_30           0
Momentum_7       0
Momentum_30      0
dtype: int64


- Dilakukan penghapusan baris yang mengandung missing value menggunakan fungsi dropna().
- Missing value muncul sebagai konsekuensi dari proses feature engineering, terutama pada fitur Lag, Moving Average, Volatility, dan - Momentum yang membutuhkan data historis dari periode sebelumnya.
- Penghapusan missing value bertujuan memastikan seluruh data yang digunakan memiliki informasi yang lengkap sehingga tidak mengganggu proses analisis maupun pemodelan.
- Setelah proses penghapusan dilakukan, dilakukan pengecekan ulang menggunakan isnull().sum() untuk memastikan tidak ada missing value yang tersisa pada dataset.
- Hasilnya, dataset menjadi bersih dan siap digunakan pada tahap seleksi fitur serta pemodelan selanjutnya.

## Tahap 3. Drop High Multicollinearity

In [44]:
corr_matrix = price_df_fe.corr().abs()
corr_matrix

# Ambil upper triangle
upper_triangle = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

In [45]:
high_corr_features = [
    column
    for column in upper_triangle.columns
    if any(upper_triangle[column] > 0.90)
]

print("Highly Correlated Features:")
print(high_corr_features)

Highly Correlated Features:
['High', 'Low', 'Close', 'Lag_1', 'Lag_7', 'Lag_30', 'MA_7', 'MA_30', 'EMA_7', 'EMA_30']


In [46]:
columns_to_drop = [
    'MA_7',
    'MA_30'
]

price_df_fe = price_df_fe.drop(
    columns=columns_to_drop
)

In [47]:
price_df_fe.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2483 entries, 30 to 2512
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           2483 non-null   datetime64[ns]
 1   Open           2483 non-null   float64       
 2   High           2483 non-null   float64       
 3   Low            2483 non-null   float64       
 4   Close          2483 non-null   float64       
 5   Volume         2483 non-null   float64       
 6   Lag_1          2483 non-null   float64       
 7   Lag_7          2483 non-null   float64       
 8   Lag_30         2483 non-null   float64       
 9   Return_1D      2483 non-null   float64       
 10  Volatility_7   2483 non-null   float64       
 11  Volatility_30  2483 non-null   float64       
 12  HL_Spread      2483 non-null   float64       
 13  OC_Diff        2483 non-null   float64       
 14  Volume_MA_7    2483 non-null   float64       
 15  Volume_MA_30   2483 non-n

- Dilakukan analisis korelasi antar fitur menggunakan correlation matrix untuk mengidentifikasi fitur yang memiliki hubungan sangat kuat satu sama lain.
- Nilai korelasi absolut digunakan agar hubungan positif maupun negatif dapat terdeteksi dengan baik.
- Selanjutnya dibuat upper triangle matrix untuk menghindari perhitungan pasangan korelasi yang sama secara berulang.
- Fitur dengan nilai korelasi lebih dari 0,90 diidentifikasi sebagai fitur yang memiliki multikolinearitas tinggi dan berpotensi memberikan informasi yang redundan.
- Hasil identifikasi menunjukkan beberapa fitur memiliki korelasi yang sangat tinggi, terutama pada kelompok Moving Average yang merepresentasikan informasi tren yang serupa.
- Untuk mengurangi redundansi informasi dan menjaga efisiensi dataset, fitur MA_7 dan MA_30 dihapus karena informasinya telah cukup terwakili oleh fitur tren lain.
- Setelah penghapusan dilakukan, dataset menjadi lebih ringkas tanpa kehilangan informasi penting yang dibutuhkan untuk analisis dan pemodelan.

## Tahap 4. Correlation-Based Selection

In [48]:
target_corr = (
    price_df_fe.corr()['Close']
    .sort_values(ascending=False)
)

print(target_corr)

Close            1.000000
Low              0.999827
High             0.999686
Open             0.999530
Lag_1            0.999388
EMA_7            0.999352
EMA_30           0.999352
Lag_7            0.996456
Lag_30           0.988605
Volatility_30    0.830751
Date             0.825940
Volatility_7     0.729994
HL_Spread        0.552766
Momentum_30      0.435697
Momentum_7       0.189441
Return_1D        0.054363
OC_Diff         -0.008922
Volume          -0.027785
Volume_MA_7     -0.067122
Volume_MA_30    -0.176109
Name: Close, dtype: float64


In [49]:
selected_features = target_corr[
    abs(target_corr) > 0.2
].index
print(selected_features)
price_df_fe = price_df_fe[selected_features]

Index(['Close', 'Low', 'High', 'Open', 'Lag_1', 'EMA_7', 'EMA_30', 'Lag_7',
       'Lag_30', 'Volatility_30', 'Date', 'Volatility_7', 'HL_Spread',
       'Momentum_30'],
      dtype='object')


- Dilakukan perhitungan korelasi setiap fitur terhadap variabel target Close menggunakan matriks korelasi.
- Nilai korelasi kemudian diurutkan untuk mengetahui fitur mana yang memiliki hubungan paling kuat dengan harga penutupan emas.
- Fitur dipilih menggunakan ambang batas |korelasi| > 0,2, sehingga hanya fitur yang memiliki hubungan cukup signifikan dengan target yang dipertahankan.
- Metode ini bertujuan mengurangi fitur yang kurang informatif sekaligus mempertahankan fitur yang memiliki kontribusi lebih besar terhadap prediksi harga emas.
- Hasil seleksi menunjukkan bahwa fitur-fitur seperti Lag, Moving Average, EMA, Volatility, Momentum, Range, dan Volume memiliki hubungan yang cukup kuat dengan harga penutupan sehingga dipertahankan dalam dataset.
- Setelah proses seleksi dilakukan, dataset menjadi lebih relevan dan efisien untuk digunakan pada tahap pemodelan tanpa kehilangan informasi penting yang berkaitan dengan target prediksi.

##Tahap 5. Save Dataset

In [50]:
price_df_fe.to_csv(
    'gold_data_fe.csv',
    index=False
)

In [51]:
price_df_fe.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2483 entries, 30 to 2512
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Close          2483 non-null   float64       
 1   Low            2483 non-null   float64       
 2   High           2483 non-null   float64       
 3   Open           2483 non-null   float64       
 4   Lag_1          2483 non-null   float64       
 5   EMA_7          2483 non-null   float64       
 6   EMA_30         2483 non-null   float64       
 7   Lag_7          2483 non-null   float64       
 8   Lag_30         2483 non-null   float64       
 9   Volatility_30  2483 non-null   float64       
 10  Date           2483 non-null   datetime64[ns]
 11  Volatility_7   2483 non-null   float64       
 12  HL_Spread      2483 non-null   float64       
 13  Momentum_30    2483 non-null   float64       
dtypes: datetime64[ns](1), float64(13)
memory usage: 291.0 KB


In [52]:
price_df_fe.head()

,Close,Low,High,Open,Lag_1,EMA_7,EMA_30,Lag_7,Lag_30,Volatility_30,Date,Volatility_7,HL_Spread,Momentum_30
30,1336.699951,1321.900024,1344.199951,1324.500000,1318.400024,1315.816595,1315.816595,1268.000000,1254.199951,37.935290,2016-07-01,24.257108,22.299927,82.500000
31,1356.400024,1339.699951,1356.800049,1342.199951,1336.699951,1325.962452,1325.962452,1261.199951,1252.400024,41.111951,2016-07-05,14.408555,17.100098,104.000000
32,1364.900024,1359.000000,1374.900024,1359.000000,1356.400024,1335.696845,1335.696845,1320.000000,1251.099976,44.325222,2016-07-06,19.541272,15.900024,113.800049
33,1360.099976,1350.500000,1368.599976,1366.800049,1364.900024,1341.797628,1341.797628,1322.500000,1228.900024,46.066637,2016-07-07,20.964596,18.099976,131.199951
34,1356.599976,1335.000000,1365.800049,1361.300049,1360.099976,1345.498215,1345.498215,1315.300049,1223.500000,46.909929,2016-07-08,18.751121,30.800049,133.099976


In [53]:
price_df_fe.describe()

,Close,Low,High,Open,Lag_1,EMA_7,EMA_30,Lag_7,Lag_30,Volatility_30,Date,Volatility_7,HL_Spread,Momentum_30
count,2483.000000,2483.000000,2483.000000,2483.000000,2483.000000,2483.000000,2483.000000,2483.000000,2483.000000,2483.000000,2483,2483.000000,2483.000000,2483.000000
mean,1977.151911,1966.264841,1987.717883,1977.049619,1975.853562,1973.170476,1973.170476,1967.770799,1935.795326,42.351830,2021-06-11 03:59:31.002819072,21.648106,21.453042,41.356585
min,1127.800049,1123.900024,1132.800049,1126.900024,1127.800049,1136.264416,1136.264416,1127.800049,1127.800049,5.971304,2016-07-01 00:00:00,1.133456,0.000000,-696.100098
25%,1323.000000,1319.850037,1327.750000,1323.800049,1322.850037,1324.381529,1324.381529,1322.250000,1319.150024,19.182763,2018-12-22 12:00:00,9.017721,7.000000,-30.200012
50%,1791.400024,1783.300049,1798.400024,1791.099976,1790.699951,1790.834199,1790.834199,1788.699951,1782.599976,30.228823,2021-06-11 00:00:00,14.090616,13.900024,16.799927
75%,2018.099976,2012.849976,2028.599976,2022.349976,2017.700012,2012.756197,2012.756197,2014.300049,1993.950012,46.791403,2023-11-28 12:00:00,24.112208,25.000000,89.400024
max,5318.399902,5301.600098,5586.200195,5415.700195,5318.399902,5186.285922,5186.285922,5318.399902,5318.399902,303.740527,2026-05-19 00:00:00,270.387055,740.500000,1011.699707
std,866.682399,857.591727,875.761789,867.327077,865.251790,861.378173,861.378173,855.826892,816.490737,42.762475,NaN,25.277361,32.641526,136.553170


Tahap terakhir dilakukan untuk mempersiapkan dataset akhir sebelum digunakan pada proses modeling oleh tim AI.

Dataset akhir yang dihasilkan telah:

- bersih dari missing value,
- memiliki fitur yang relevan,
- siap digunakan untuk pelatihan model LSTM dan GRU.